## Cleaning Dataset: Analisis Sentimen Sumbawa Hijau Lestari

**Tujuan:** Membersihkan data mentah hasil scraping Apify Facebook Comments Scraper (436 baris, 243 kolom) menjadi dataset yang siap dilabeli secara manual untuk analisis sentimen.

**Tahapan:**
1. Load CSV mentah
2. Seleksi kolom relevan
3. Filter baris tidak valid (error, teks kosong)
4. Deduplikasi
5. Standardisasi teks & tanggal
6. Mapping nama sumber (54 postingan)
7. Tandai jenis komentar & kandidat spam
8. Susun kolom final & export ke Excel
9. Ringkasan statistik cleaning

---
## 0. Import Library & Konfigurasi Path

In [1]:
import pandas as pd
import numpy as np
import re
import unicodedata
from pathlib import Path

# ── Path konfigurasi ─────────────────────────────────────────────────────────
BASE_DIR    = Path('..')                      # root proyek
RAW_CSV     = BASE_DIR / 'data' / 'dataset' / 'dataset_analisis_sumbawa_hijau_lestari.csv'
SUMBER_CSV  = BASE_DIR / 'data' / 'dataset' / 'daftar_link_sumber.csv'
OUTPUT_XLSX = BASE_DIR / 'data' / 'processed_dataset' / 'dataset_siap_label_sumbawa_hijau_lestari.xlsx'

print('Direktori proyek :', BASE_DIR)
print('File CSV mentah  :', RAW_CSV.exists(), '--', RAW_CSV)
print('File daftar link :', SUMBER_CSV.exists(), '--', SUMBER_CSV)
print('Output akan ke   :', OUTPUT_XLSX)

Direktori proyek : ..
File CSV mentah  : True -- ..\data\dataset\dataset_analisis_sumbawa_hijau_lestari.csv
File daftar link : True -- ..\data\dataset\daftar_link_sumber.csv
Output akan ke   : ..\data\processed_dataset\dataset_siap_label_sumbawa_hijau_lestari.xlsx


---
## 1. Load File CSV Mentah

In [2]:
df_raw = pd.read_csv(RAW_CSV, low_memory=False)

print(f'Shape data mentah : {df_raw.shape}  ({df_raw.shape[0]} baris, {df_raw.shape[1]} kolom)')
df_raw.head(3)

Shape data mentah : (436, 243)  (436 baris, 243 kolom)


,attachments/0/__typename,attachments/0/animated_image,attachments/0/animated_image/height,attachments/0/animated_image/uri,attachments/0/animated_image/width,attachments/0/animated_image_attribution,attachments/0/animated_image_caption,attachments/0/blurred_image/uri,attachments/0/cix_screen,attachments/0/fallback_image/height,...,parentReply/author/name,parentReply/id,postTitle,profileId,profileName,profilePicture,profileUrl,replyToCommentId,text,threadingDepth
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,🌱✨ Sumbawa Hijau Lestari 2026 ✨🌱\nSmpnsatu Lab...,100003800193447,Bunda Shifa Dan Khanza,https://scontent-bos5-1.xx.fbcdn.net/v/t39.308...,https://www.facebook.com/erni.shifaamirin,NaN,Mntap pak guru luar biasa,0.0
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,SEMANGAT HUT KE-81 REPUBLIK INDONESIA! ✨🇮🇩\n\n...,pfbid0ms6oD9tSJEKnhvCMTy4pa44thYRHvbmqotiuLegN...,Aj Syamsinar J In,https://scontent-dfw5-2.xx.fbcdn.net/v/t39.308...,https://www.facebook.com/ajsyamsinar.jin,NaN,Masyaallah bunda 2solehah dan cantik semuanya ...,0.0


---
## 2. Seleksi Kolom yang Relevan

In [3]:
KOLOM_RELEVAN = [
    'date',             # Tanggal komentar (ISO 8601 UTC)
    'profileName',      # Nama akun yang berkomentar
    'text',             # Isi komentar
    'inputUrl',         # URL postingan sumber
    'likesCount',       # Jumlah like (untuk EDA nanti)
    'threadingDepth',   # 0=utama, >=1=balasan
    'replyToCommentId', # ID komentar induk (jika balasan)
    'error',            # Terisi 'no_items' jika post private/gagal fetch
]

df = df_raw[KOLOM_RELEVAN].copy()
n_awal = len(df)

print(f'Jumlah baris awal : {n_awal}')
print()
print('Nilai null per kolom:')
display(df.isnull().sum().rename('Null Count').to_frame())

Jumlah baris awal : 436

Nilai null per kolom:


,Null Count
date,5
profileName,5
text,34
inputUrl,0
likesCount,5
threadingDepth,5
replyToCommentId,268
error,431


---
## 3. Filter Baris Tidak Valid

### 3a. Buang baris dengan kolom `error` terisi (post private / gagal fetch)

In [4]:
mask_error = df['error'].notna()
print(f'Baris dengan error  : {mask_error.sum()} baris')
print(df[mask_error][['inputUrl', 'error']].to_string())

df = df[~mask_error].copy()
print(f'\nBaris setelah buang error : {len(df)}')

Baris dengan error  : 5 baris
                                         inputUrl     error
1    https://web.facebook.com/share/p/1CdKbco55h/  no_items
36   https://web.facebook.com/share/p/1DCUyjGA8K/  no_items
79   https://web.facebook.com/share/p/1CM3YpRNeh/  no_items
158  https://web.facebook.com/share/p/1Jc7amLBwS/  no_items
159  https://web.facebook.com/share/p/1BmBSTdvG4/  no_items

Baris setelah buang error : 431


### 3b. Buang baris dengan `text` kosong / NaN (sticker, GIF, reaksi tanpa teks)

In [5]:
mask_no_text = df['text'].isna() | (df['text'].astype(str).str.strip() == '')
print(f'Baris tanpa teks (sticker/GIF/kosong) : {mask_no_text.sum()} baris')

df = df[~mask_no_text].copy()
print(f'Baris setelah buang teks kosong       : {len(df)}')

Baris tanpa teks (sticker/GIF/kosong) : 29 baris
Baris setelah buang teks kosong       : 402


---
## 4. Deduplikasi

Hapus baris dengan kombinasi `(profileName, text, inputUrl)` yang identik persis (redundansi struktur export Apify).

In [6]:
n_sebelum_dedup = len(df)

# Tampilkan duplikat untuk verifikasi
dup_mask = df.duplicated(subset=['profileName', 'text', 'inputUrl'], keep=False)
n_dup_rows = dup_mask.sum()
if n_dup_rows > 0:
    print(f'Baris yang terlibat duplikat : {n_dup_rows}')
    display(
        df[dup_mask][['profileName', 'text', 'inputUrl', 'date']]
        .sort_values(['profileName', 'text'])
    )
else:
    print('Tidak ada duplikat ditemukan.')

# Hapus duplikat — pertahankan kemunculan pertama
df = df.drop_duplicates(subset=['profileName', 'text', 'inputUrl'], keep='first').copy()
print(f'\nBaris sebelum dedup : {n_sebelum_dedup}')
print(f'Baris setelah dedup : {len(df)}')
print(f'Baris dihapus       : {n_sebelum_dedup - len(df)}')

Baris yang terlibat duplikat : 2


,profileName,text,inputUrl,date
63,Aini Andriani,Terimakasih atas dukungannya..semoga kita bisa...,https://web.facebook.com/share/r/1Hs2WJvAni/,2026-01-06T04:29:54.000Z
64,Aini Andriani,Terimakasih atas dukungannya..semoga kita bisa...,https://web.facebook.com/share/r/1Hs2WJvAni/,2026-01-06T04:29:54.000Z



Baris sebelum dedup : 402
Baris setelah dedup : 401
Baris dihapus       : 1


---
## 5. Standardisasi Teks Komentar

> **Catatan:** Bahasa Sumbawa (Basa Samawa) dan Bahasa Indonesia **TIDAK** diterjemahkan/dinormalisasi maknanya.
> Hanya dirapikan: encoding unicode, literal `\\n`, dan spasi berlebih.  
> Emoji **dipertahankan** karena relevan sebagai sinyal sentimen.

In [7]:
def standardize_text(text):
    """Rapikan teks komentar tanpa mengubah makna.

    - Normalisasi unicode NFC (menyatukan karakter komposit)
    - Ganti literal backslash-n (dua karakter) menjadi spasi
    - Ganti newline/carriage return nyata menjadi spasi
    - Kompres whitespace berlebih menjadi satu spasi
    - Trim spasi di awal dan akhir
    """
    if not isinstance(text, str):
        return text

    # 1. Normalisasi unicode ke NFC
    text = unicodedata.normalize('NFC', text)

    # 2. Ganti literal \n (dua karakter: \ dan n) => spasi
    text = text.replace('\\n', ' ')

    # 3. Ganti newline dan carriage return nyata => spasi
    text = text.replace('\r\n', ' ').replace('\n', ' ').replace('\r', ' ')

    # 4. Kompres whitespace ganda (spasi, tab) => satu spasi
    text = re.sub(r'[ \t]+', ' ', text)

    # 5. Trim
    return text.strip()


df['text'] = df['text'].apply(standardize_text)

print('Standardisasi teks selesai.')
print('Contoh 5 teks pertama:')
for i, t in enumerate(df['text'].head(5).tolist(), 1):
    print(f'  [{i}] {t}')

Standardisasi teks selesai.
Contoh 5 teks pertama:
  [1] Mntap pak guru luar biasa
  [2] Masyaallah bunda 2solehah dan cantik semuanya tetap semangat 👍👍
  [3] Seruuu!!!! maju terus TP PKK Pekat
  [4] Mantap pak kadis.. semangat🔥
  [5] Kereen memulaii gerakan hebat Sumbawa lestari. Top


---
## 6. Standardisasi Tanggal

Konversi dari ISO 8601 UTC (`2026-06-10T02:53:30.000Z`) ke:
- Kolom `Tanggal` -> format `YYYY-MM-DD`
- Kolom `Waktu` -> format `HH:MM:SS` UTC (opsional untuk EDA waktu)

In [8]:
df['date_parsed'] = pd.to_datetime(df['date'], utc=True, errors='coerce')

df['Tanggal'] = df['date_parsed'].dt.strftime('%Y-%m-%d')
df['Waktu']   = df['date_parsed'].dt.strftime('%H:%M:%S')  # UTC

# Cek tanggal yang gagal parse
n_gagal_parse = df['date_parsed'].isna().sum()
if n_gagal_parse > 0:
    print(f'Peringatan: {n_gagal_parse} baris gagal parse tanggal:')
    display(df[df['date_parsed'].isna()][['date', 'profileName', 'text']])
else:
    print('Semua tanggal berhasil diparse.')

print(f"Rentang tanggal: {df['Tanggal'].min()} s.d. {df['Tanggal'].max()}")
df[['date', 'Tanggal', 'Waktu']].head()

Semua tanggal berhasil diparse.
Rentang tanggal: 2025-12-16 s.d. 2026-08-19


,date,Tanggal,Waktu
0,2026-06-10T02:53:30.000Z,2026-06-10,02:53:30
2,2026-08-15T15:24:16.000Z,2026-08-15,15:24:16
4,2026-08-15T21:49:01.000Z,2026-08-15,21:49:01
6,2025-12-16T04:46:08.000Z,2025-12-16,04:46:08
7,2025-12-16T05:02:26.000Z,2025-12-16,05:02:26


---
## 7. Mapping Nama Sumber

Buat kolom baru **`Nama Sumber`** berdasarkan mapping `inputUrl` ke `Nama Sumber` dari tabel referensi `daftar_link_sumber.csv`.

> **Langkah manual yang diperlukan:** Buka file `data/dataset/daftar_link_sumber.csv`
> dan isi kolom **Nama Sumber** dengan nama akun Facebook yang sebenarnya,
> lalu jalankan ulang sel ini.

In [9]:
df_sumber = pd.read_csv(SUMBER_CSV)
print(f'Jumlah entri di daftar_link_sumber.csv : {len(df_sumber)}')
display(df_sumber.head(10))

Jumlah entri di daftar_link_sumber.csv : 54


,No,Nama Sumber,Link
0,1,Ahmad Yani,https://web.facebook.com/share/p/14mVZWWrhmp/
1,2,Sekretaris Daerah Kabupaten Sumbawa,https://web.facebook.com/share/p/18qWSBaLH9/
2,3,Adhe Panglima,https://web.facebook.com/share/p/18wcRrYMr4/
3,4,Iwan Sofian,https://web.facebook.com/share/p/192S9tuKvA/
4,5,Ardi Ardi,https://web.facebook.com/share/p/193hfpsbPD/
5,6,Patimah Zulkarnaen,https://web.facebook.com/share/p/19JBjuFtv9/
6,7,Muhammad Taufan,https://web.facebook.com/share/p/19RSkiGJAa/
7,8,Prokopim Sumbawa,https://web.facebook.com/share/p/19WR82h7VL/
8,9,Fitrah Berliansyah,https://web.facebook.com/share/p/19WV3GsF78/
9,10,Yahdil Fady,https://web.facebook.com/share/p/19XeG2JEhB/


In [10]:
# Buat dictionary URL => Nama Sumber
url_to_nama = dict(zip(df_sumber['Link'].str.strip(), df_sumber['Nama Sumber'].str.strip()))

# Mapping ke kolom baru
df['Nama Sumber'] = df['inputUrl'].str.strip().map(url_to_nama)

# Verifikasi URL yang tidak ter-mapping
unmapped = df[df['Nama Sumber'].isna()]['inputUrl'].unique()
if len(unmapped) > 0:
    print(f'Peringatan: {len(unmapped)} URL tidak ter-mapping ke Nama Sumber:')
    for u in unmapped:
        print(f'  {u}')
else:
    print('Semua URL berhasil di-mapping ke Nama Sumber.')

print('\nDistribusi komentar per Nama Sumber:')
display(df['Nama Sumber'].value_counts().rename_axis('Nama Sumber').reset_index(name='Jumlah Komentar'))

Semua URL berhasil di-mapping ke Nama Sumber.

Distribusi komentar per Nama Sumber:


,Nama Sumber,Jumlah Komentar
0,Rahmat Hidayat,25
1,Iwan Sofian,23
2,Abdul Hakim,21
3,Zakariah Surbini,20
4,Abdul Azis,20
5,Kominfotiksandi Pemkab Sumbawa,20
6,Yanti Flora,20
7,Berlian Sumbawa,19
8,Ida Nursanti,17
9,Samawa Lawyer Club,16


---
## 8. Tandai Jenis Komentar

- `threadingDepth == 0` -> **"Utama"**
- `threadingDepth >= 1` -> **"Balasan"**

In [11]:
df['Jenis Komentar'] = df['threadingDepth'].apply(
    lambda x: 'Utama' if pd.notna(x) and int(x) == 0 else 'Balasan'
)

print('Distribusi Jenis Komentar:')
display(df['Jenis Komentar'].value_counts().rename_axis('Jenis Komentar').reset_index(name='Jumlah'))

Distribusi Jenis Komentar:


,Jenis Komentar,Jumlah
0,Utama,234
1,Balasan,167


---
## 9. Tandai Kandidat Spam

Buat kolom boolean `Kemungkinan_Spam = True` jika teks mengandung pola promosi/spam.  
**Tidak dihapus otomatis** — ditandai untuk direview manual saat pelabelan.

In [12]:
SPAM_KEYWORDS = [
    # Kata kunci promosi umum
    r'\bjasa\b', r'\border\b', r'\bpromo\b', r'\bjual\b', r'\bdijual\b',
    r'\bmurah\b', r'\bharga\b', r'\bgrosir\b', r'\bdiskon\b', r'\bdiscount\b',
    r'\bgratis\b', r'\bproduk\b', r'\bshop\b', r'\bstore\b', r'\btoko\b',
    r'\bwholesale\b',
    # CTA promosi
    r'\bDM\b', r'\bwa\.?me\b', r'\bwhatsapp\b',
    r'\bhubungi\b', r'\bcontact\b', r'\bpesan\s*(?:sekarang|langsung)\b',
    # Link eksternal non-Facebook
    r'https?://(?!(?:web\.)?facebook\.com|fb\.com|fb\.watch)\S+',
    r'bit\.ly/', r'tinyurl\.com/', r'linktr\.ee/',
    # Nomor HP dengan konteks promosi
    r'(?:wa|telp|hp|tlp|call)[\s:]*(?:\+?62|0)8\d{7,11}',
]

SPAM_PATTERN = re.compile('|'.join(SPAM_KEYWORDS), flags=re.IGNORECASE)

df['Kemungkinan_Spam'] = df['text'].apply(
    lambda t: bool(SPAM_PATTERN.search(t)) if isinstance(t, str) else False
)

n_spam = df['Kemungkinan_Spam'].sum()
print(f'Kandidat spam ditemukan : {n_spam} baris')
if n_spam > 0:
    print('\nDaftar kandidat spam (untuk review manual):')
    display(
        df[df['Kemungkinan_Spam']][['profileName', 'text', 'Nama Sumber', 'Tanggal']]
        .reset_index(drop=True)
    )

Kandidat spam ditemukan : 10 baris

Daftar kandidat spam (untuk review manual):


,profileName,text,Nama Sumber,Tanggal
0,Iwan Sofian,Saya menawarkan videogram di Storefront saya! ...,Iwan Sofian,2026-06-21
1,Nurul Ulo,yuk saling support guys https://www.facebook.c...,Nurul Ulo,2026-06-06
2,Aini Andriani,Terimakasih atas dukungannya..semoga kita bisa...,Aini Andriani,2026-01-06
3,Wahab Penyaring,Tidak ada usaha yang sia sia Tak ada usaha tam...,Wahab Penyaring,2026-01-28
4,Iwan Sofian,Saya menawarkan videogram di Storefront saya! ...,Iwan Sofian,2026-01-08
5,Yanti Flora,https://www.facebook.com/yanti.flora.2025/subs...,Yanti Flora,2026-07-31
6,Ida Nursanti,Dukung saya dengan memberi bintang https://www...,Ida Nursanti,2025-12-28
7,Abdul Azis,"terima kasih dukungan bintang 🌟 nya , dengan a...",Abdul Azis,2026-06-05
8,Abdul Hakim,https://samawaijo.blogspot.com/?m=0,Abdul Hakim,2026-06-06
9,Yosy Larian,Coba hubungi pak kades ne duan?,Yosy Larian,2025-12-28


---
## 10. Susun Kolom Final (Template Pelabelan)

| No | Tanggal | Platform/Sumber | Nama Sumber | Nama Akun | Isi Komentar | Jenis Komentar | Link Sumber | Kemungkinan_Spam | Kategori Isu | Sentimen |
|----|---------|----------------|-------------|-----------|--------------|----------------|-------------|-----------------|--------------|----------|

- `Platform/Sumber` diisi konstan `"Facebook"`
- `Kategori Isu` dan `Sentimen` dikosongkan `""` — akan diisi manual pada tahap pelabelan

In [13]:
df_final = df[[
    'Tanggal',
    'Nama Sumber',
    'profileName',
    'text',
    'Jenis Komentar',
    'inputUrl',
    'Kemungkinan_Spam',
]].copy()

# Tambahkan kolom konstan dan kolom label kosong
df_final.insert(0, 'No', range(1, len(df_final) + 1))
df_final.insert(2, 'Platform/Sumber', 'Facebook')
df_final['Kategori Isu'] = ''
df_final['Sentimen']     = ''

# Rename kolom sesuai template
df_final = df_final.rename(columns={
    'profileName' : 'Nama Akun',
    'text'        : 'Isi Komentar',
    'inputUrl'    : 'Link Sumber',
})

# Urutan kolom final
KOLOM_FINAL = [
    'No', 'Tanggal', 'Platform/Sumber', 'Nama Sumber',
    'Nama Akun', 'Isi Komentar', 'Jenis Komentar',
    'Link Sumber', 'Kemungkinan_Spam',
    'Kategori Isu', 'Sentimen',
]
df_final = df_final[KOLOM_FINAL]

print(f'Shape dataframe final : {df_final.shape}')
print(f'Kolom                 : {df_final.columns.tolist()}')
df_final.head(5)

Shape dataframe final : (401, 11)
Kolom                 : ['No', 'Tanggal', 'Platform/Sumber', 'Nama Sumber', 'Nama Akun', 'Isi Komentar', 'Jenis Komentar', 'Link Sumber', 'Kemungkinan_Spam', 'Kategori Isu', 'Sentimen']


,No,Tanggal,Platform/Sumber,Nama Sumber,Nama Akun,Isi Komentar,Jenis Komentar,Link Sumber,Kemungkinan_Spam,Kategori Isu,Sentimen
0,1,2026-06-10,Facebook,Fitrah Berliansyah,Bunda Shifa Dan Khanza,Mntap pak guru luar biasa,Utama,https://web.facebook.com/share/p/19WV3GsF78/,False,,
2,2,2026-08-15,Facebook,TP PKK Kel Pekat,Aj Syamsinar J In,Masyaallah bunda 2solehah dan cantik semuanya ...,Utama,https://web.facebook.com/share/p/1ECFD78AQK/,False,,
4,3,2026-08-15,Facebook,TP PKK Kel Pekat,Syamsiah Saleh,Seruuu!!!! maju terus TP PKK Pekat,Utama,https://web.facebook.com/share/p/1ECFD78AQK/,False,,
6,4,2025-12-16,Facebook,Ahmad Yani,Jufrie Karri,Mantap pak kadis.. semangat🔥,Utama,https://web.facebook.com/share/p/14mVZWWrhmp/,False,,
7,5,2025-12-16,Facebook,Ahmad Yani,Suhardi Soud,Kereen memulaii gerakan hebat Sumbawa lestari....,Utama,https://web.facebook.com/share/p/14mVZWWrhmp/,False,,


---
## 11. Export ke Excel (`.xlsx`)

In [14]:
OUTPUT_XLSX.parent.mkdir(parents=True, exist_ok=True)

with pd.ExcelWriter(OUTPUT_XLSX, engine='openpyxl') as writer:
    # Sheet 1: Dataset utama siap label
    df_final.to_excel(writer, index=False, sheet_name='Dataset Siap Label')

    # Sheet 2: Kandidat spam untuk review manual
    df_spam = df_final[df_final['Kemungkinan_Spam'] == True].copy()
    df_spam.to_excel(writer, index=False, sheet_name='Review Spam')

    # Sheet 3: Log statistik cleaning
    log_data = {
        'Keterangan': [
            'Total baris mentah (raw)',
            'Dibuang: error (post private/gagal fetch)',
            'Dibuang: teks kosong/NaN (sticker, GIF, dll)',
            'Dibuang: duplikat (profileName + text + inputUrl)',
            'Total baris bersih (final)',
            '---',
            'Komentar Utama (threadingDepth=0)',
            'Balasan (threadingDepth>=1)',
            '---',
            'Ditandai Kemungkinan_Spam',
        ],
        'Jumlah': [
            n_awal,
            mask_error.sum(),
            mask_no_text.sum(),
            n_sebelum_dedup - len(df),
            len(df_final),
            '',
            (df_final['Jenis Komentar'] == 'Utama').sum(),
            (df_final['Jenis Komentar'] == 'Balasan').sum(),
            '',
            int(df_final['Kemungkinan_Spam'].sum()),
        ]
    }
    pd.DataFrame(log_data).to_excel(writer, index=False, sheet_name='Log Cleaning')

print('File Excel berhasil disimpan ke:')
print(f'   {OUTPUT_XLSX}')

File Excel berhasil disimpan ke:
   ..\data\processed_dataset\dataset_siap_label_sumbawa_hijau_lestari.xlsx


---
## 12. Ringkasan Statistik Cleaning

In [15]:
print('=' * 60)
print('RINGKASAN CLEANING -- SUMBAWA HIJAU LESTARI')
print('=' * 60)

print('\nSEBELUM CLEANING')
print(f'   Total baris mentah          : {n_awal}')

print('\nBARIS YANG DIBUANG')
print(f'   Error (post private/gagal)  : {mask_error.sum()}')
print(f'   Teks kosong (sticker/GIF)   : {mask_no_text.sum()}')
print(f'   Duplikat                    : {n_sebelum_dedup - len(df)}')

print('\nSETELAH CLEANING')
print(f'   Total baris bersih          : {len(df_final)}')

print('\nDISTRIBUSI JENIS KOMENTAR')
vc_jenis = df_final['Jenis Komentar'].value_counts()
for jenis, jml in vc_jenis.items():
    print(f'   {jenis:<12} : {jml} ({jml/len(df_final)*100:.1f}%)')

print('\nKANDIDAT SPAM')
print(f"   Ditandai Kemungkinan_Spam   : {df_final['Kemungkinan_Spam'].sum()}")
print('   Lihat sheet "Review Spam" di file Excel untuk verifikasi manual')

print('\nOUTPUT')
print(f'   {OUTPUT_XLSX}')
print(f"   Sheet 1 : Dataset Siap Label ({len(df_final)} baris)")
print(f"   Sheet 2 : Review Spam ({df_final['Kemungkinan_Spam'].sum()} baris)")
print('   Sheet 3 : Log Cleaning')
print('=' * 60)

RINGKASAN CLEANING -- SUMBAWA HIJAU LESTARI

SEBELUM CLEANING
   Total baris mentah          : 436

BARIS YANG DIBUANG
   Error (post private/gagal)  : 5
   Teks kosong (sticker/GIF)   : 29
   Duplikat                    : 1

SETELAH CLEANING
   Total baris bersih          : 401

DISTRIBUSI JENIS KOMENTAR
   Utama        : 234 (58.4%)
   Balasan      : 167 (41.6%)

KANDIDAT SPAM
   Ditandai Kemungkinan_Spam   : 10
   Lihat sheet "Review Spam" di file Excel untuk verifikasi manual

OUTPUT
   ..\data\processed_dataset\dataset_siap_label_sumbawa_hijau_lestari.xlsx
   Sheet 1 : Dataset Siap Label (401 baris)
   Sheet 2 : Review Spam (10 baris)
   Sheet 3 : Log Cleaning


---
## Langkah Selanjutnya

1. **Isi Nama Sumber** — Buka `data/dataset/daftar_link_sumber.csv`, isi kolom `Nama Sumber`
   dengan nama akun Facebook yang sebenarnya dari setiap link, lalu jalankan ulang notebook ini.
2. **Review spam** — Buka sheet `Review Spam` di file Excel, verifikasi manual apakah setiap
   baris memang spam atau false positive. Hapus manual jika dikonfirmasi spam.
3. **Pelabelan manual** — Isi kolom `Kategori Isu` dan `Sentimen` di sheet `Dataset Siap Label`:
   - `Sentimen`: Positif / Negatif / Netral
   - `Kategori Isu`: misalnya Penghijauan, Pengelolaan Sampah, Daur Ulang, Apresiasi, Kritik, dll.
4. **Langkah berikutnya** -> `02_eda_dataset.ipynb` (Exploratory Data Analysis)